# Airflow DAG runs
Which workflows (DAGs) ran recently in Airflow, which failed, and the failed tasks' logs. Runs against the notebook's environment: your local stack, or a deployed one (dev, stage), which it only reads. Pick it on the Notebooks page, or call `target("dev")`.

Locally it signs in with your stack's Airflow login (from the stack checkout's `.env`, else `developer` / `123456`). For dev or stage, set `BLUECORE_DEV_AIRFLOW_USER` and `…_PASSWORD` (or `…_AIRFLOW_TOKEN`) in `.env.notebooks`.

In [ ]:
# Your local stack, or dev / stage: pick it on the Notebooks page.
from bluecore import DASHBOARD_API, show, target
bc = target()
bc

In [ ]:
# Airflow 3 (api/v2) signs in for a token; Airflow 2 (api/v1) takes the login on each request
s, API = bc.airflow()
print("Using", API, "-", s.get(f"{API}/version").json())

## DAGs

In [ ]:
import pandas as pd
dags = s.get(f"{API}/dags", params={"limit": 100}, timeout=15).json()["dags"]
pd.DataFrame([{"dag": d["dag_id"], "paused": d.get("is_paused"), "tags": ", ".join(t["name"] for t in d.get("tags") or [])} for d in dags])

## Recent runs of every DAG

In [ ]:
runs = []
for d in dags:
    r = s.get(f"{API}/dags/{d['dag_id']}/dagRuns", params={"limit": 5, "order_by": "-start_date"}, timeout=15)
    for run in r.json().get("dag_runs", []):
        runs.append({"dag": d["dag_id"], "run": run["dag_run_id"], "state": run["state"], "start": run.get("start_date"), "end": run.get("end_date")})
runs = pd.DataFrame(runs)
runs.sort_values("start", ascending=False).head(25) if not runs.empty else "No runs yet"

## Why a run failed
The failed tasks of the latest failed run, with the end of each task's log.

In [ ]:
failed = runs[runs["state"] == "failed"] if not runs.empty else runs
if failed.empty:
    print("No failed runs in the recent history.")
else:
    run = failed.sort_values("start", ascending=False).iloc[0]
    print(f"{run['dag']} / {run['run']}")
    tis = s.get(f"{API}/dags/{run['dag']}/dagRuns/{run['run']}/taskInstances", timeout=15).json()["task_instances"]
    for ti in [t for t in tis if t["state"] == "failed"]:
        log = s.get(f"{API}/dags/{run['dag']}/dagRuns/{run['run']}/taskInstances/{ti['task_id']}/logs/{ti.get('try_number', 1)}",
                    headers={"Accept": "text/plain"}, timeout=30)
        print(f"\n--- {ti['task_id']} (try {ti.get('try_number')}) ---")
        print(log.text[-3000:])